# AgriSense — Data Preparation & Cleaning

This notebook prepares the PlantSeg dataset for machine learning.

The raw dataset was first investigated in `01_eda.ipynb`.

The EDA identified several data-quality issues, including duplicate images, conflicting disease labels, cross-split duplicates, class imbalance, and varying image dimensions.

The goal of this notebook is to create a reliable dataset for model training while avoiding data leakage and preserving as much valid information as possible.

In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
import hashlib

In [2]:
#Loading the metadata

#Define the project root
project_root = Path.cwd().parent

#Define important data paths
metadata_path = project_root / "data" / "raw" / "PlantSeg"
image_dir = metadata_path / "images"

#Load the metadata
metadata_file = metadata_path / "metadata.csv"

df=pd.read_csv(metadata_file)

print("project root" , project_root)
print("Image Directory",image_dir)
print("metadata Shape",df.shape)
print("Metadata columns",df.columns.tolist())

project root c:\Users\Dell\Documents\Projects\AgriSense
Image Directory c:\Users\Dell\Documents\Projects\AgriSense\data\raw\PlantSeg\images
metadata Shape (7774, 10)
Metadata columns ['Name', 'Index', 'Plant', 'Disease', 'Resolution', 'Label file', 'Mask ratio', 'URL', 'License', 'Split']


In [3]:
#initial record values

print("Initial Dataset size : ",len(df))
print("Number of plants : ", df["Plant"].nunique())
print("Number of Diseases : ",df["Disease"].nunique())

print("Split Distribution : \n",df["Split"].value_counts())
print("Missing values : ", df.isnull().sum().sum())

Initial Dataset size :  7774
Number of plants :  34
Number of Diseases :  115
Split Distribution : 
 Split
Training      5367
Test          1561
Validation     846
Name: count, dtype: int64
Missing values :  0


In [4]:
#create image hashes 

def calculate_md5(file_path):
    """returns the MD5 hash of an image file"""

    with open(file_path,"rb") as f:
        return hashlib.md5(f.read()).hexdigest()

print("Hash function created successfuly!")

Hash function created successfuly!


In [5]:
image_lookup = {}

for image_path in image_dir.rglob("*"):

    if image_path.is_file():
        image_lookup[image_path.name] = image_path

print("Images found = ",len(image_lookup))

Images found =  7774


In [6]:
# Calculate the MD5 hash for every image

df["image_hash"] = df["Name"].map(
    lambda filename: calculate_md5(image_lookup[filename])
)

print("Hashes calculated:", df["image_hash"].notna().sum())
print("Unique image contents:", df["image_hash"].nunique())

Hashes calculated: 7774
Unique image contents: 7497


In [7]:
#Finding image contents that occur mor than once

duplicate_counts = df["image_hash"].value_counts()

duplicate_hashes = duplicate_counts[
    duplicate_counts > 1
].index

duplicate_records = df[
    df["image_hash"].isin(duplicate_hashes)
].sort_values("image_hash")

print("Records belonging to duplicate groups:", len(duplicate_records))
print("Number of duplicate image groups:", len(duplicate_hashes))

Records belonging to duplicate groups: 542
Number of duplicate image groups: 265


In [8]:
#Summarize each duplicate image group

duplicate_summary = (
    duplicate_records
    .groupby("image_hash")
    .agg(
        records = ("Name","count"),
        diseases = ("Disease","nunique"),
        plants = ("Plant","nunique"),
        splits = ("Split","nunique")
    )
    .reset_index()
)

print(duplicate_summary.head())

print("Number of duplicate group : ",len(duplicate_summary))

                         image_hash  records  diseases  plants  splits
0  010baf1a0f09318a09c9b144d4447242        2         2       1       2
1  0187ea51a7e765edfc02691ce1f3c737        2         2       2       2
2  0251ad19b65433696a01f07aa21197bd        2         1       1       2
3  03672baf1a488539501ffaa8f151c7d5        2         1       1       2
4  0688b47a55dd1f46873b84e5567631e4        2         2       2       1
Number of duplicate group :  265


In [9]:
#categorize duplicate image groups by disease and split consistency

same_disease = duplicate_summary["diseases"] == 1
same_plant = duplicate_summary["plants"] == 1
same_split = duplicate_summary["splits"] == 1

print("Duplicate groups with same disease:",
      same_disease.sum())

print("Duplicate groups with conflicting diseases:",
      (~same_disease).sum())

print("Duplicate groups within one split:",
      same_split.sum())

print("Duplicate groups across multiple splits:",
      (~same_split).sum())

Duplicate groups with same disease: 167
Duplicate groups with conflicting diseases: 98
Duplicate groups within one split: 142
Duplicate groups across multiple splits: 123


In [11]:
#Identify image contents with conflicting disease labels

conflicting_hashes = duplicate_summary.loc[
    duplicate_summary["diseases"] > 1,
    "image_hash"
]

print("Conflicting image contents : ",len(conflicting_hashes))

Conflicting image contents :  98


In [12]:
#creating a decision table for duplicate image groups

duplicate_decisions = duplicate_summary.copy()

duplicate_decisions["conflicting_disease"] =(duplicate_decisions["diseases"] > 1) 

duplicate_decisions["conflicting_plant"] = (duplicate_decisions["plants"] > 1)

duplicate_decisions["cross_split"] = (duplicate_decisions["splits"] > 1)

print(duplicate_decisions.head())

                         image_hash  records  diseases  plants  splits  \
0  010baf1a0f09318a09c9b144d4447242        2         2       1       2   
1  0187ea51a7e765edfc02691ce1f3c737        2         2       2       2   
2  0251ad19b65433696a01f07aa21197bd        2         1       1       2   
3  03672baf1a488539501ffaa8f151c7d5        2         1       1       2   
4  0688b47a55dd1f46873b84e5567631e4        2         2       2       1   

   conflicting_disease  conflicting_plant  cross_split  
0                 True              False         True  
1                 True               True         True  
2                False              False         True  
3                False              False         True  
4                 True               True        False  


In [13]:
# Cross-tabulate duplicate groups by disease consistency and split consistency

decision_matrix = pd.crosstab(
    duplicate_decisions["conflicting_disease"],
    duplicate_decisions["cross_split"]
)

print(decision_matrix)

cross_split          False  True 
conflicting_disease              
False                   96     71
True                    46     52


In [14]:
#check for duplicate image content with conflicting plant labels

plant_conflict_only = duplicate_decisions.loc[
    (duplicate_decisions["conflicting_plant"]) & 
    (~duplicate_decisions["conflicting_disease"])
]

print("Duplicate groups with conflicting plant labels but consistent disease labels : ",len(plant_conflict_only))

Duplicate groups with conflicting plant labels but consistent disease labels :  0


In [15]:
#counting all records belonging to conflicting plant labels

conflicting_records = df[
    df["image_hash"].isin(conflicting_hashes)
]

print("Conflicting image contents : ",len(conflicting_records))
print("records to be excluded : ",len(conflicting_records))

Conflicting image contents :  205
records to be excluded :  205


In [17]:
#creating a copy containing only records with consistent labels

df_clean = df[
    ~df["image_hash"].isin(conflicting_hashes)
].copy()

print("Original df : ", len(df))
print("df after removing conflicts ", len(df_clean))
print("records removed : ", len(df) - len(df_clean))

Original df :  7774
df after removing conflicts  7569
records removed :  205


In [18]:
# Find duplicate image contents remaining after conflict removal

remaining_duplicate_counts = df_clean["image_hash"].value_counts()

remaining_duplicate_hashes = remaining_duplicate_counts[
    remaining_duplicate_counts > 1
].index

print("Remaining duplicate groups:",
      len(remaining_duplicate_hashes))

print("Records belonging to remaining duplicate groups:",
      df_clean["image_hash"].isin(remaining_duplicate_hashes).sum())

Remaining duplicate groups: 167
Records belonging to remaining duplicate groups: 337


In [19]:
# Analyze the remaining duplicate groups by split

remaining_duplicate_summary = (
    df_clean[df_clean["image_hash"].isin(remaining_duplicate_hashes)]
    .groupby("image_hash")
    .agg(
        records=("Name", "count"),
        diseases=("Disease", "nunique"),
        plants=("Plant", "nunique"),
        splits=("Split", "nunique")
    )
    .reset_index()
)

remaining_duplicate_summary["cross_split"] = (
    remaining_duplicate_summary["splits"] > 1
)

print(
    "Duplicate groups within the same split:",
    (~remaining_duplicate_summary["cross_split"]).sum()
)

print(
    "Duplicate groups across multiple splits:",
    remaining_duplicate_summary["cross_split"].sum()
)

Duplicate groups within the same split: 96
Duplicate groups across multiple splits: 71


In [20]:
#Inspect the split assignment of the 71 cross split duplicate groups 

cross_split_groups = remaining_duplicate_summary[
    remaining_duplicate_summary["cross_split"]
].copy()

cross_split_details = (
    df_clean[
    df_clean["image_hash"].isin(cross_split_groups["image_hash"])
    ]
    .groupby("image_hash")
    .agg(
        files = ("Name", list),
        plants = ("Plant", list),
        diseases = ("Disease", list),
        splits = ("Split", list)
    )
    .reset_index()
)

print("Cross-split duplicate groups:", len(cross_split_details))

display(cross_split_details.head(10))

Cross-split duplicate groups: 71


,image_hash,files,plants,diseases,splits
0,0251ad19b65433696a01f07aa21197bd,"[banana_cigar_end_rot_Google_0010.jpg, banana_...","[Banana, Banana]","[banana cigar end rot, banana cigar end rot]","[Training, Test]"
1,03672baf1a488539501ffaa8f151c7d5,"[cabbage_black_rot_Bing_0013.jpg, cabbage_blac...","[Cabbage, Cabbage]","[cabbage black rot, cabbage black rot]","[Training, Test]"
2,20775ae5d1e073db18b49db0f428839a,"[peach_scab_Bing_0253.jpg, peach_scab_Bing_022...","[Peach, Peach]","[peach scab, peach scab]","[Training, Test]"
3,22e694652b9ecca2262289a65abe880b,"[coffee_brown_eye_spot_Google_0034.jpg, coffee...","[Coffee, Coffee]","[coffee brown eye spot, coffee brown eye spot]","[Training, Test]"
4,2668102d15ca6d355bff4c1b56ff073f,"[wheat_septoria_blotch_Google_0338.jpg, wheat_...","[Wheat, Wheat]","[wheat septoria blotch, wheat septoria blotch]","[Training, Validation]"
5,272a0041449565fff936657171a14c1f,"[wheat_stripe_rust_Bing_0031.jpg, wheat_stripe...","[Wheat, Wheat]","[wheat stripe rust, wheat stripe rust]","[Training, Test]"
6,2acad4b70da681219737958028755ae1,[wheat_septoria_blotch_google_septoria (13).jp...,"[Wheat, Wheat]","[wheat septoria blotch, wheat septoria blotch]","[Training, Test]"
7,2c44d7a8fca422c1d60f3d293b6d0aff,"[wheat_stripe_rust_Bing_0063.jpg, wheat_stripe...","[Wheat, Wheat]","[wheat stripe rust, wheat stripe rust]","[Validation, Training]"
8,2e40141bcec55909eed169f5b820f603,"[grape_downy_mildew_122.jpg, grape_downy_milde...","[Grape, Grape]","[grape downy mildew, grape downy mildew]","[Training, Test]"
9,378d63b7fd017daa31123d67a7342e16,"[soybean_frog_eye_leaf_spot_Bing_0039.jpg, soy...","[Soybean, Soybean]","[soybean frog eye leaf spot, soybean frog eye ...","[Training, Test]"


In [21]:
# Assign one final split to every remaining image content

split_priority = {
    "Test": 3,
    "Validation": 2,
    "Training": 1
}

def choose_final_split(splits):
    """Choose the highest-priority split for an image."""
    return max(splits, key=lambda split: split_priority[split])


cross_split_details["final_split"] = (
    cross_split_details["splits"]
    .apply(choose_final_split)
)

display(
    cross_split_details[
        ["image_hash", "splits", "final_split"]
    ].head(10)
)

,image_hash,splits,final_split
0,0251ad19b65433696a01f07aa21197bd,"[Training, Test]",Test
1,03672baf1a488539501ffaa8f151c7d5,"[Training, Test]",Test
2,20775ae5d1e073db18b49db0f428839a,"[Training, Test]",Test
3,22e694652b9ecca2262289a65abe880b,"[Training, Test]",Test
4,2668102d15ca6d355bff4c1b56ff073f,"[Training, Validation]",Validation
5,272a0041449565fff936657171a14c1f,"[Training, Test]",Test
6,2acad4b70da681219737958028755ae1,"[Training, Test]",Test
7,2c44d7a8fca422c1d60f3d293b6d0aff,"[Validation, Training]",Validation
8,2e40141bcec55909eed169f5b820f603,"[Training, Test]",Test
9,378d63b7fd017daa31123d67a7342e16,"[Training, Test]",Test


In [22]:
# Create a final split lookup for the cross-split duplicate groups

final_split_lookup = dict(
    zip(
        cross_split_details["image_hash"],
        cross_split_details["final_split"]
    )
)

# Assign the final split to every remaining record
df_clean["final_split"] = df_clean["image_hash"].map(
    final_split_lookup
)

# For images that were not cross-split duplicates,
# keep their original split
df_clean["final_split"] = df_clean["final_split"].fillna(
    df_clean["Split"]
)

print(df_clean["final_split"].value_counts())

final_split
Training      5171
Test          1564
Validation     834
Name: count, dtype: int64
